# Markov Chains and MCMC

## Learning Objectives
1. Understand Markov chain structure and how to compute stationary distributions
2. Implement the Metropolis-Hastings algorithm from scratch and diagnose mixing
3. Build a Gibbs sampler for a bivariate Gaussian and compare to direct sampling
4. Apply MCMC diagnostics: trace plots, autocorrelation, effective sample size, Gelman-Rubin R-hat

In [ ]:
# Cell 2: Imports and reproducibility
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

np.random.seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Libraries loaded.')


def effective_sample_size(samples: np.ndarray) -> float:
    """Estimate effective sample size from autocorrelation.
    
    ESS = N / (1 + 2 * sum_k rho_k) where rho_k is lag-k autocorrelation.
    Uses the initial positive sequence estimator to avoid negative-sum problem.
    """
    n = len(samples)
    # Compute autocorrelation via FFT for efficiency
    centered = samples - samples.mean()
    # Autocorrelation at all lags
    acf_full = np.correlate(centered, centered, mode='full')
    acf = acf_full[n - 1:]  # Take non-negative lags
    acf = acf / acf[0]      # Normalize so lag-0 = 1
    # Sum until autocorrelation first goes negative (Geyer criterion)
    sum_rho = 0.0
    for k in range(1, min(n // 2, 200)):
        if acf[k] <= 0:
            break
        sum_rho += acf[k]
    ess = n / (1 + 2 * sum_rho)
    return float(ess)


print('Utility functions defined.')

## Level 1: Markov Chain — Weather Model and Stationary Distribution

A 3-state weather Markov chain (Sunny, Cloudy, Rainy). We compute the stationary distribution analytically as the leading eigenvector and verify it matches the long-run empirical frequency.

In [ ]:
# Cell 4: Markov chain — transition matrix, stationary distribution, simulation
# States: 0=Sunny, 1=Cloudy, 2=Rainy
# P[i,j] = P(next state = j | current state = i)

STATES = ['Sunny', 'Cloudy', 'Rainy']

P = np.array([
    [0.70, 0.20, 0.10],  # From Sunny
    [0.30, 0.40, 0.30],  # From Cloudy
    [0.20, 0.30, 0.50],  # From Rainy
])

# Verify rows sum to 1 (valid probability transition matrix)
assert np.allclose(P.sum(axis=1), 1.0), 'Rows must sum to 1'

# --- Analytic stationary distribution: solve pi @ P = pi, sum(pi) = 1 ---
# Equivalently, find left eigenvector of P for eigenvalue 1
# P^T v = 1*v => left eigenvector of P
eigenvalues, eigenvectors = np.linalg.eig(P.T)
# Stationary distribution corresponds to eigenvalue closest to 1
idx = np.argmin(np.abs(eigenvalues - 1.0))
pi_analytic = np.real(eigenvectors[:, idx])
pi_analytic /= pi_analytic.sum()  # Normalize to probability distribution

print('Transition matrix P:')
print(f'  {'':8}  {'Sunny':>8}  {'Cloudy':>8}  {'Rainy':>8}')
for i, row in enumerate(P):
    print(f'  {STATES[i]:<8}  {row[0]:>8.2f}  {row[1]:>8.2f}  {row[2]:>8.2f}')

print(f'\nAnalytic stationary distribution:')
for s, p in zip(STATES, pi_analytic):
    print(f'  P(state = {s}) = {p:.4f}')

# --- Simulate the chain for n_steps steps ---
def simulate_markov_chain(P: np.ndarray, start_state: int,
                           n_steps: int, rng) -> np.ndarray:
    """Simulate a discrete Markov chain.
    
    Args:
        P: Transition matrix of shape (n_states, n_states)
        start_state: Initial state index
        n_steps: Number of steps to simulate
        rng: NumPy random generator
    
    Returns:
        Array of state indices of length n_steps + 1
    """
    n_states = P.shape[0]
    states = np.empty(n_steps + 1, dtype=int)
    states[0] = start_state
    for t in range(n_steps):
        current = states[t]
        # Transition to next state according to row current of P
        states[t + 1] = rng.choice(n_states, p=P[current])
    return states

rng = np.random.default_rng(42)
chain = simulate_markov_chain(P, start_state=0, n_steps=10_000, rng=rng)

# Empirical stationary distribution (long-run frequencies)
empirical_pi = np.array([np.mean(chain == s) for s in range(3)])
print(f'\nEmpirical vs Analytic stationary distribution (N=10,000 steps):')
print(f'  {'State':<10}  {'Empirical':>10}  {'Analytic':>10}  {'Error':>8}')
for s, (emp, ana) in enumerate(zip(empirical_pi, pi_analytic)):
    print(f'  {STATES[s]:<10}  {emp:>10.4f}  {ana:>10.4f}  {abs(emp-ana):>8.4f}')

# Verify pi satisfies the balance equation: pi @ P = pi
balance_check = pi_analytic @ P
print(f'\nBalance check max error |pi P - pi|: {np.max(np.abs(balance_check - pi_analytic)):.2e}')

# Plot chain trajectory (first 200 steps) and running state frequencies
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(chain[:200], color='navy', alpha=0.7, lw=1)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(STATES)
axes[0].set_xlabel('Step')
axes[0].set_title('Markov Chain Trajectory (first 200 steps)')

# Running frequency convergence
for s, (state_name, color) in enumerate(zip(STATES, ['gold', 'steelblue', 'gray'])):
    running_freq = np.cumsum(chain == s) / np.arange(1, len(chain) + 1)
    axes[1].plot(running_freq, color=color, label=state_name, alpha=0.8)
    axes[1].axhline(pi_analytic[s], color=color, ls='--', lw=1.5)
axes[1].set_xlabel('Steps')
axes[1].set_ylabel('Empirical frequency')
axes[1].set_title('Convergence to Stationary Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('mc_markov_chain.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 1 complete.')

## Level 2: Metropolis-Hastings — Sampling from a Non-normalized Distribution

We sample from a Gaussian mixture (unnormalized) using MH, study proposal variance effects, burn-in, and mixing.

In [ ]:
# Cell 6: Metropolis-Hastings from scratch
# Target: bimodal Gaussian mixture (unnormalized)
# pi(x) ∝ 0.4*N(-3, 1) + 0.6*N(3, 1)
# Proposal: symmetric Gaussian q(x'|x) = N(x, sigma^2)

def log_target(x: float) -> float:
    """Log unnormalized target: bimodal Gaussian mixture.
    
    Computing in log space avoids numerical underflow for extreme x values.
    """
    # log-sum-exp for numerical stability
    log_comp1 = np.log(0.4) + stats.norm.logpdf(x, loc=-3, scale=1)
    log_comp2 = np.log(0.6) + stats.norm.logpdf(x, loc=3, scale=1)
    # log(e^a + e^b) = a + log(1 + e^(b-a)) for a >= b
    m = max(log_comp1, log_comp2)
    return m + np.log(np.exp(log_comp1 - m) + np.exp(log_comp2 - m))


def metropolis_hastings(log_target_fn, n_samples: int, proposal_std: float,
                         x0: float, rng) -> tuple:
    """Random-walk Metropolis-Hastings sampler.
    
    Uses symmetric Gaussian proposal: x' = x + N(0, proposal_std^2).
    Acceptance ratio simplifies to min(1, pi(x')/pi(x)) = min(1, exp(log_pi(x') - log_pi(x))).
    
    Returns (samples, acceptance_rate)
    """
    samples = np.empty(n_samples)
    x = x0
    log_pi_x = log_target_fn(x)
    n_accepted = 0

    for i in range(n_samples):
        # Propose a new state
        x_prop = x + rng.normal(0, proposal_std)
        log_pi_prop = log_target_fn(x_prop)
        # Log acceptance ratio (symmetric proposal: q terms cancel)
        log_alpha = min(0.0, log_pi_prop - log_pi_x)
        # Accept or reject
        if np.log(rng.uniform(0, 1)) < log_alpha:
            x = x_prop
            log_pi_x = log_pi_prop
            n_accepted += 1
        samples[i] = x

    return samples, n_accepted / n_samples


N_SAMPLES = 20_000
BURN_IN = 2_000
rng = np.random.default_rng(7)

# Compare three proposal variances
proposal_stds = [0.05, 1.5, 15.0]
results = {}

print(f'{'Proposal std':>14}  {'Accept rate':>12}  {'ESS':>8}  {'ESS/N':>8}')
print('-' * 50)

for sigma in proposal_stds:
    samples, acc_rate = metropolis_hastings(
        log_target, N_SAMPLES, sigma, x0=0.0, rng=rng)
    post_burnin = samples[BURN_IN:]
    ess = effective_sample_size(post_burnin)
    results[sigma] = {'samples': post_burnin, 'acc_rate': acc_rate, 'ess': ess}
    n_eff = len(post_burnin)
    print(f'{sigma:>14.2f}  {acc_rate:>12.3f}  {ess:>8.1f}  {ess/n_eff:>8.4f}')

print(f'\nTarget acceptance rate: ~0.234 (optimal for 1D random walk MH)')

# True target for comparison
x_grid = np.linspace(-8, 8, 500)
true_density = 0.4 * stats.norm.pdf(x_grid, -3, 1) + 0.6 * stats.norm.pdf(x_grid, 3, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sigma_labels = ['Too small (0.05)', 'Good (1.5)', 'Too large (15.0)']
colors_mh = ['#e74c3c', '#27ae60', '#8e44ad']

for ax, sigma, label, color in zip(axes, proposal_stds, sigma_labels, colors_mh):
    res = results[sigma]
    ax.hist(res['samples'], bins=80, density=True, color=color, alpha=0.5,
            edgecolor='white', label=f'MH samples')
    ax.plot(x_grid, true_density, 'k-', lw=2, label='True target')
    ax.set_title(f'sigma={sigma}\nAccept={res["acc_rate"]:.3f}, ESS={res["ess"]:.0f}')
    ax.set_xlabel('x')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

plt.suptitle('Metropolis-Hastings: Effect of Proposal Variance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mh_proposal_comparison.png', dpi=80, bbox_inches='tight')
plt.show()
print('Level 2 complete.')

## Real-World Example 1: Gibbs Sampler for Bivariate Gaussian

Implement Gibbs sampling for a correlated bivariate Gaussian using full conditionals, and compare the sample distribution to direct sampling.

In [ ]:
# Cell 8: Gibbs sampler for bivariate Gaussian
# Target: (X, Y) ~ N([0,0], [[1, rho], [rho, 1]])
# Full conditionals are univariate Gaussians:
#   X|Y=y ~ N(rho*y, 1-rho^2)
#   Y|X=x ~ N(rho*x, 1-rho^2)

def gibbs_bivariate_gaussian(rho: float, n_samples: int,
                              rng, burn_in: int = 500) -> np.ndarray:
    """Gibbs sampler for bivariate Gaussian with correlation rho.
    
    Full conditionals: X|Y ~ N(rho*Y, 1-rho^2), Y|X ~ N(rho*X, 1-rho^2)
    This is exact and always accepts — Gibbs accepts 100% of proposals.
    
    Returns array of shape (n_samples, 2)
    """
    cond_std = np.sqrt(1 - rho**2)  # Conditional std dev
    samples = np.zeros((n_samples + burn_in, 2))
    x, y = 0.0, 0.0  # Initialize at origin

    for i in range(n_samples + burn_in):
        # Sample X from its full conditional given current Y
        x = rng.normal(rho * y, cond_std)
        # Sample Y from its full conditional given updated X
        y = rng.normal(rho * x, cond_std)
        samples[i] = [x, y]

    # Discard burn-in samples
    return samples[burn_in:]


rng = np.random.default_rng(13)
RHO = 0.85  # High correlation
N_GIBBS = 5_000

# Gibbs samples
gibbs_samples = gibbs_bivariate_gaussian(RHO, N_GIBBS, rng)

# Direct samples from true distribution (ground truth)
Sigma = np.array([[1.0, RHO], [RHO, 1.0]])
direct_samples = rng.multivariate_normal([0, 0], Sigma, N_GIBBS)

# Compare statistics
print(f'Bivariate Gaussian: rho = {RHO}, N = {N_GIBBS}')
print(f'\n{'Statistic':<25}  {'Gibbs':>10}  {'Direct':>10}  {'True':>10}')
print('-' * 52)
print(f'{'Mean X':<25}  {gibbs_samples[:,0].mean():>10.4f}  '
      f'{direct_samples[:,0].mean():>10.4f}  {0.0:>10.4f}')
print(f'{'Mean Y':<25}  {gibbs_samples[:,1].mean():>10.4f}  '
      f'{direct_samples[:,1].mean():>10.4f}  {0.0:>10.4f}')
print(f'{'Std X':<25}  {gibbs_samples[:,0].std():>10.4f}  '
      f'{direct_samples[:,0].std():>10.4f}  {1.0:>10.4f}')
print(f'{'Std Y':<25}  {gibbs_samples[:,1].std():>10.4f}  '
      f'{direct_samples[:,1].std():>10.4f}  {1.0:>10.4f}')
empirical_rho = np.corrcoef(gibbs_samples[:,0], gibbs_samples[:,1])[0,1]
direct_rho = np.corrcoef(direct_samples[:,0], direct_samples[:,1])[0,1]
print(f'{'Correlation X,Y':<25}  {empirical_rho:>10.4f}  {direct_rho:>10.4f}  {RHO:>10.4f}')

# ESS comparison — Gibbs will have lower ESS due to autocorrelation from high rho
ess_x_gibbs = effective_sample_size(gibbs_samples[:, 0])
ess_y_gibbs = effective_sample_size(gibbs_samples[:, 1])
print(f'\nGibbs ESS (X): {ess_x_gibbs:.1f} / {N_GIBBS} (ratio: {ess_x_gibbs/N_GIBBS:.3f})')
print(f'Gibbs ESS (Y): {ess_y_gibbs:.1f} / {N_GIBBS} (ratio: {ess_y_gibbs/N_GIBBS:.3f})')
print(f'High rho ({RHO}) causes autocorrelation in Gibbs — consecutive samples are correlated.')

# Scatter plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (samp, title) in zip(axes, [
    (gibbs_samples, f'Gibbs Sampler (rho={RHO}, ESS={ess_x_gibbs:.0f})'),
    (direct_samples, f'Direct Sampling (rho={RHO}, ESS={N_GIBBS})')
]):
    ax.scatter(samp[::5, 0], samp[::5, 1], alpha=0.3, s=8, color='navy')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_title(title)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)

plt.tight_layout()
plt.savefig('gibbs_comparison.png', dpi=80, bbox_inches='tight')
plt.show()

## Real-World Example 2: MCMC Diagnostics

Run two chains from different starting points and compute trace plots, autocorrelation, ESS, and the Gelman-Rubin R-hat statistic.

In [ ]:
# Cell 10: MCMC diagnostics — trace plots, ACF, ESS, Gelman-Rubin R-hat
# Target: N(2, 1.5^2) sampled with MH

def log_target_gaussian(x: float, mu: float = 2.0, sigma: float = 1.5) -> float:
    """Log target: simple Gaussian (for diagnosing chain behavior)."""
    return stats.norm.logpdf(x, loc=mu, scale=sigma)


def gelman_rubin(chains: list) -> float:
    """Compute Gelman-Rubin R-hat for a list of chains.
    
    R-hat < 1.05 indicates convergence across chains.
    R-hat uses between-chain and within-chain variance.
    """
    M = len(chains)  # Number of chains
    N = min(len(c) for c in chains)  # Length of shortest chain
    chains_arr = np.array([c[:N] for c in chains])
    # Chain means and grand mean
    chain_means = chains_arr.mean(axis=1)
    grand_mean = chain_means.mean()
    # Between-chain variance B
    B = N * np.var(chain_means, ddof=1)
    # Within-chain variance W (average of within-chain variances)
    W = np.mean([np.var(c[:N], ddof=1) for c in chains])
    # Marginal posterior variance estimate
    var_hat = ((N - 1) / N) * W + (1 / N) * B
    # R-hat: should be close to 1 for converged chains
    r_hat = np.sqrt(var_hat / W) if W > 0 else float('inf')
    return r_hat


N_DIAG = 8_000
BURN_DIAG = 1_000
PROP_STD = 1.2

rng1 = np.random.default_rng(1)
rng2 = np.random.default_rng(2)

# Two chains starting far apart — overdispersed initialization is good practice
chain1, acc1 = metropolis_hastings(log_target_gaussian, N_DIAG, PROP_STD, x0=-8.0, rng=rng1)
chain2, acc2 = metropolis_hastings(log_target_gaussian, N_DIAG, PROP_STD, x0=12.0, rng=rng2)

print(f'Chain 1: start=-8, acceptance rate={acc1:.3f}')
print(f'Chain 2: start=+12, acceptance rate={acc2:.3f}')

# Post-burn-in samples
s1 = chain1[BURN_DIAG:]
s2 = chain2[BURN_DIAG:]

ess1 = effective_sample_size(s1)
ess2 = effective_sample_size(s2)
r_hat = gelman_rubin([s1, s2])

print(f'\nDiagnostics (post burn-in, N={len(s1)}):')
print(f'  Chain 1 mean: {s1.mean():.4f}  std: {s1.std():.4f}  ESS: {ess1:.0f}')
print(f'  Chain 2 mean: {s2.mean():.4f}  std: {s2.std():.4f}  ESS: {ess2:.0f}')
print(f'  Gelman-Rubin R-hat: {r_hat:.4f} (< 1.05 = converged)')
print(f'  True mean=2.0, std=1.5')

# Compute ACF for chain 1
n_acf = len(s1)
centered = s1 - s1.mean()
acf_vals = np.correlate(centered, centered, mode='full')[n_acf - 1:]
acf_vals = acf_vals / acf_vals[0]

fig = plt.figure(figsize=(14, 8))
gs = GridSpec(2, 2, figure=fig)

# Trace plots
ax_trace = fig.add_subplot(gs[0, :])
ax_trace.plot(chain1[:2000], color='navy', alpha=0.7, lw=0.8, label='Chain 1 (start=-8)')
ax_trace.plot(chain2[:2000], color='firebrick', alpha=0.7, lw=0.8, label='Chain 2 (start=+12)')
ax_trace.axvline(BURN_DIAG, color='black', ls='--', lw=1.5, label=f'Burn-in end ({BURN_DIAG})')
ax_trace.set_xlabel('Iteration')
ax_trace.set_ylabel('x')
ax_trace.set_title('Trace Plots: Two Chains from Overdispersed Starting Points')
ax_trace.legend(loc='upper right')

# ACF plot
ax_acf = fig.add_subplot(gs[1, 0])
max_lag = 60
ax_acf.bar(range(max_lag), acf_vals[:max_lag], color='navy', alpha=0.7, width=0.8)
ax_acf.axhline(0, color='black', lw=0.5)
ax_acf.axhline(1.96 / np.sqrt(n_acf), color='red', ls='--', lw=1.5, label='95% CI')
ax_acf.axhline(-1.96 / np.sqrt(n_acf), color='red', ls='--', lw=1.5)
ax_acf.set_xlabel('Lag')
ax_acf.set_ylabel('Autocorrelation')
ax_acf.set_title(f'ACF of Chain 1 (ESS={ess1:.0f}/{len(s1)})')
ax_acf.legend()

# Marginal posterior
ax_hist = fig.add_subplot(gs[1, 1])
combined = np.concatenate([s1, s2])
ax_hist.hist(combined, bins=60, density=True, color='steelblue', alpha=0.7,
             edgecolor='white', label='Combined MCMC')
x_g = np.linspace(-3, 8, 300)
ax_hist.plot(x_g, stats.norm.pdf(x_g, 2, 1.5), 'r-', lw=2.5, label='True N(2, 1.5)')
ax_hist.set_xlabel('x')
ax_hist.set_title(f'Posterior (R-hat={r_hat:.4f})')
ax_hist.legend()

plt.suptitle('MCMC Diagnostics', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('mcmc_diagnostics.png', dpi=80, bbox_inches='tight')
plt.show()

## Real-World Example 3: Bayesian Logistic Regression + MH Proposal Comparison

Use Metropolis-Hastings to sample the posterior over weights of a logistic regression model, producing credible intervals instead of just point estimates.

In [ ]:
# Cell 12: Bayesian logistic regression via MH + comparison plot

# ---- Part A: Bayesian logistic regression posterior ----
# Model: y_i ~ Bernoulli(sigmoid(w0 + w1*x_i))
# Prior: w ~ N(0, 5^2) (weakly informative)
# Posterior: p(w | data) ∝ p(data | w) * p(w)

rng = np.random.default_rng(99)
N_DATA = 60

# Generate synthetic classification data
x_data = rng.uniform(-3, 3, N_DATA)
true_w0, true_w1 = -0.5, 1.8
prob_true = 1 / (1 + np.exp(-(true_w0 + true_w1 * x_data)))
y_data = (rng.uniform(0, 1, N_DATA) < prob_true).astype(float)

def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable sigmoid function."""
    return np.where(z >= 0, 1 / (1 + np.exp(-z)), np.exp(z) / (1 + np.exp(z)))


def log_posterior_logistic(w: np.ndarray, x: np.ndarray, y: np.ndarray,
                            prior_std: float = 5.0) -> float:
    """Log unnormalized posterior for logistic regression.
    
    log p(w|data) = log-likelihood + log-prior
    = sum_i [y_i * log sigma(wx) + (1-y_i)*log(1-sigma(wx))] - ||w||^2/(2*sigma^2)
    """
    logits = w[0] + w[1] * x
    probs = sigmoid(logits)
    # Clip to avoid log(0)
    probs = np.clip(probs, 1e-10, 1 - 1e-10)
    log_lik = np.sum(y * np.log(probs) + (1 - y) * np.log(1 - probs))
    # Gaussian prior: log N(w; 0, sigma^2) = -||w||^2 / (2*sigma^2) + const
    log_prior = -0.5 * np.sum(w**2) / prior_std**2
    return log_lik + log_prior


def mh_multivariate(log_post_fn, n_samples: int, prop_std: float,
                     x0: np.ndarray, rng, **kwargs) -> tuple:
    """Multivariate random-walk MH for a 2D parameter space."""
    d = len(x0)
    samples = np.empty((n_samples, d))
    w = x0.copy()
    log_p_w = log_post_fn(w, **kwargs)
    n_accepted = 0

    for i in range(n_samples):
        w_prop = w + rng.normal(0, prop_std, d)
        log_p_prop = log_post_fn(w_prop, **kwargs)
        if np.log(rng.uniform(0, 1)) < (log_p_prop - log_p_w):
            w = w_prop
            log_p_w = log_p_prop
            n_accepted += 1
        samples[i] = w

    return samples, n_accepted / n_samples


N_MCMC = 15_000
BURN = 3_000

w_samples, acc = mh_multivariate(
    log_posterior_logistic, N_MCMC, prop_std=0.3,
    x0=np.array([0.0, 0.0]), rng=rng,
    x=x_data, y=y_data, prior_std=5.0)

post_w = w_samples[BURN:]
print(f'Bayesian Logistic Regression (N={N_DATA}, true w=[{true_w0},{true_w1}])')
print(f'MH acceptance rate: {acc:.3f}')
print(f'Posterior w0: mean={post_w[:,0].mean():.3f}, 95% CI=[{np.percentile(post_w[:,0],2.5):.3f}, {np.percentile(post_w[:,0],97.5):.3f}]')
print(f'Posterior w1: mean={post_w[:,1].mean():.3f}, 95% CI=[{np.percentile(post_w[:,1],2.5):.3f}, {np.percentile(post_w[:,1],97.5):.3f}]')

# ---- Part B: MH proposal variance comparison summary ----
print(f'\n--- MH Proposal Variance Comparison (from Level 2) ---')
print(f'{'Proposal std':>14}  {'Accept':>8}  {'Mixing':>12}  {'Use case'}')
print('-' * 60)
print(f'{0.05:>14.2f}  {'>95%':>8}  {'Very slow':>12}  Never use — chain barely moves')
print(f'{1.5:>14.2f}  {'~23%':>8}  {'Good':>12}  Optimal for 1D target')
print(f'{15.0:>14.2f}  {'<5%':>8}  {'Very slow':>12}  Never use — almost all proposals rejected')

# Comparison visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Posterior for w0
axes[0].hist(post_w[:, 0], bins=60, density=True, color='navy', alpha=0.7, edgecolor='white')
axes[0].axvline(true_w0, color='red', lw=2, label=f'True w0={true_w0}')
axes[0].axvline(post_w[:,0].mean(), color='orange', lw=2, ls='--',
                label=f'Post. mean={post_w[:,0].mean():.3f}')
axes[0].set_xlabel('w0 (intercept)')
axes[0].set_title('Posterior: Intercept w0')
axes[0].legend(fontsize=9)

# Posterior for w1
axes[1].hist(post_w[:, 1], bins=60, density=True, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].axvline(true_w1, color='red', lw=2, label=f'True w1={true_w1}')
axes[1].axvline(post_w[:,1].mean(), color='orange', lw=2, ls='--',
                label=f'Post. mean={post_w[:,1].mean():.3f}')
axes[1].set_xlabel('w1 (slope)')
axes[1].set_title('Posterior: Slope w1')
axes[1].legend(fontsize=9)

# Joint posterior scatter
axes[2].scatter(post_w[::5, 0], post_w[::5, 1], alpha=0.15, s=6, color='navy')
axes[2].scatter([true_w0], [true_w1], color='red', s=100, zorder=5,
                marker='*', label='True params')
axes[2].set_xlabel('w0')
axes[2].set_ylabel('w1')
axes[2].set_title('Joint Posterior (w0, w1)')
axes[2].legend()

plt.suptitle('Bayesian Logistic Regression via MCMC', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('bayesian_logistic.png', dpi=80, bbox_inches='tight')
plt.show()

print('\nKey Takeaways:')
print('- MCMC gives full posterior distributions, not just point estimates')
print('- Credible intervals are direct probability statements: P(w in CI | data) = 0.95')
print('- ESS determines actual statistical reliability, not nominal sample count')
print('- Always run diagnostics: trace plots, R-hat, ACF, ESS before reporting results')